# Drop Detector
A compact pipeline to predict the most likely drop times in EDM tracks.

Init → Helpers → Dataset → Tuning → Training → Inference

## Init

Defines global settings and seeds; imports libs; initializes a joblib cache to speed up feature/segmentation reuse.

- Paths: BASE_DIR, CACHE_DIR

- Data/balance: dev_n_tracks, NEG_RATIO (negatives per positive), tol_label (tight training window), tol_eval (looser eval window)

- Audio: SR (sample rate), AUDIO_EXTS

- Repro: RANDOM_SEED, memory = joblib.Memory(...)

In [ ]:
# Variables
BASE_DIR = "./training_dataset"
CACHE_DIR = "./drop_detector_cache"
dev_n_tracks = 100  # subset size for hyperparameter tuning
NEG_RATIO = 4  # neg:pos cap per track
tol_label = 1.5  # tight positive window for training (s)
tol_eval = 3.0  # acceptance window for evaluation (s)

SR = 22050
AUDIO_EXTS = {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg", ".aiff", ".aif"}
RANDOM_SEED = 42

print("Dataset      :", BASE_DIR)
print("Cache dir    :", CACHE_DIR)
print("Dev cap      :", dev_n_tracks)

Dataset      : /Users/Administrateur/Software/Genre Classifier/drop detection training
Cache dir    : ./drop_detector_cache
Dev cap      : 100


In [2]:
# Init
import os, re
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import librosa
from joblib import Memory, Parallel, delayed
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
import struct
from itertools import product

os.makedirs(CACHE_DIR, exist_ok=True)
memory = Memory(location=CACHE_DIR, verbose=0)
np.random.seed(RANDOM_SEED)

try:
    from mutagen import File as MutagenFile
except Exception:
    MutagenFile = None
    warnings.warn("mutagen not available; ID3 Comment extraction will be skipped.")

## Helper functions

### File parsing

- parse_mmss(text): parse mm:ss labels from comments into seconds.

- find_audio_files(root): recursively collect audio files.

- read_sidecar_csv(root): locate/load a sidecar CSV of labels if present.

- get_comment_for_file(path, df): resolve the track’s Comment from CSV (preferred) or from file metadata (via mutagen). This is where labels come from.

### Segmentation

- load_audio_cached(path, sr): cached mono load.

- bass_boost_simple(y, sr, gain_db): lightweight FFT-domain low-band boost to emphasize bass-region cues.

- segment_track_cached(audio_path, seg_s, …): unsupervised boundary proposal via chroma+MFCC features and librosa.segment.agglomerative. Returns candidate boundary times.

### Features definition

- _segment_slice(...): takes a fixed-length window around a center time (with edge-safe padding).

- _stats_vec(x): lightweight stats helper for time series (used inside features).

- features_context_cached(audio_path, center_s, win_s, …): builds a context feature around each candidate time: pre/mid/post windows → log-mel + simple stats → concatenated vector.

### Build dataset

- build_dataset_from_pairs(pairs, seg_s, win, tol_label, tol_eval, boost_db, neg_ratio=NEG_RATIO, ...)
For each labeled (track, drop_s):

  - Segment track to get candidate boundaries.

  - Label candidates: +1 if within tol_label, -1 ignore-zone if within (tol_label, tol_eval], 0 otherwise.

  - Balance negatives: cap to neg_ratio × (#positives) to avoid class imbalance.

  - Extract context features and return stacked X, y.

### Scoring

- f1_at_two(pairs, clf, ...): score all candidates, take top-2 drops; count a hit if any is within tol_eval of the ground truth. Compute mean F1.
> Rationale: many tracks have ≥2 valid drops; F1@2 rewards finding one of them.

In [3]:
# File parsing
def parse_mmss(text):
    if not isinstance(text, str):
        return None
    m = re.search(r"(\b\d{1,2}):(\d{2})\b", text)
    if not m:
        return None
    mm, ss = int(m.group(1)), int(m.group(2))
    return mm * 60 + ss


def find_audio_files(root):
    return sorted([p for p in Path(root).rglob("*") if p.suffix.lower() in AUDIO_EXTS])


def read_sidecar_csv(root):
    for name in ("metadata.csv", "labels.csv"):
        p = Path(root) / name
        if p.exists():
            return pd.read_csv(p), p
    return None, None


def get_comment_for_file(path, df):
    """
    Return the 'comment' metadata for an audio file, preferring a CSV/DF mapping when provided.
    """
    import struct

    # 1) If a CSV/DF was provided, prefer that mapping
    if df is not None:
        cols = list(df.columns)
        cols_l = [c.lower() for c in cols]

        file_col = None
        for guess in ("filename", "file", "filepath", "path"):
            if guess in cols_l:
                file_col = cols[cols_l.index(guess)]
                break

        comment_col = None
        for guess in ("comment", "comments", "Comment"):
            if guess.lower() in cols_l:
                comment_col = cols[cols_l.index(guess.lower())]
                break

        if comment_col is None:
            return None

        fname = path.name
        if file_col is not None:
            m = df[df[file_col].astype(str).str.endswith(fname, na=False)]
            if len(m) >= 1:
                return str(m.iloc[0][comment_col])

        m = df[
            df.apply(
                lambda row: any(str(row[c]).endswith(fname) for c in df.columns), axis=1
            )
        ]
        if len(m) >= 1:
            return str(m.iloc[0][comment_col])
        return None

    # 2) No DF: read from file metadata
    # WAV special-case: RIFF parser for LIST/INFO ICMT + BWF 'bext'
    ext = path.suffix.lower()
    if ext in (".wav", ".wave"):
        try:
            with open(path, "rb") as f:
                header = f.read(12)
                if len(header) == 12:
                    riff, size, wave = struct.unpack("<4sI4s", header)
                    if riff == b"RIFF" and wave == b"WAVE":
                        while True:
                            hdr = f.read(8)
                            if len(hdr) < 8:
                                break
                            chunk_id, chunk_size = struct.unpack("<4sI", hdr)
                            pad = chunk_size & 1

                            if chunk_id == b"LIST":
                                data = f.read(chunk_size)
                                if len(data) < chunk_size:
                                    break
                                if len(data) >= 4 and data[:4] == b"INFO":
                                    i = 4
                                    while i + 8 <= len(data):
                                        sid = data[i : i + 4]
                                        ssize = struct.unpack(
                                            "<I", data[i + 4 : i + 8]
                                        )[0]
                                        start = i + 8
                                        end = start + ssize
                                        if end > len(data):
                                            break
                                        sval = data[start:end]
                                        if sid in (b"ICMT", b"COMM", b"CMNT"):
                                            txt = (
                                                sval.rstrip(b"\x00")
                                                .decode("utf-8", errors="ignore")
                                                .strip()
                                            )
                                            if not txt:
                                                txt = (
                                                    sval.rstrip(b"\x00")
                                                    .decode("latin-1", errors="ignore")
                                                    .strip()
                                                )
                                            if txt:
                                                return txt
                                        i = end + (ssize & 1)

                            elif chunk_id == b"bext":
                                data = f.read(chunk_size)
                                if len(data) < chunk_size:
                                    break
                                desc = data[:256].rstrip(b"\x00")
                                txt = desc.decode("utf-8", errors="ignore").strip()
                                if not txt:
                                    txt = desc.decode(
                                        "latin-1", errors="ignore"
                                    ).strip()
                                if txt:
                                    return txt
                            else:
                                f.seek(chunk_size + pad, 1)
        except Exception:
            pass

    # 3) Fallback to Mutagen (ID3 in WAV or other formats)
    if "MutagenFile" not in globals() or MutagenFile is None:
        return None

    try:
        mf = MutagenFile(str(path))
        if mf is None:
            return None

        for k in ("COMM::XXX", "COMM::eng", "comment", "©cmt"):
            if k in mf:
                v = mf.get(k)
                v = v[0] if isinstance(v, list) and v else v
                if v is not None:
                    return str(v)

        if getattr(mf, "tags", None):
            for k, v in (mf.tags or {}).items():
                sk = str(k).lower()
                if "comm" in sk or "comment" in sk or "icmt" in sk:
                    return str(v)
    except Exception:
        return None

    return None

In [4]:
# Segmentation
@memory.cache
def load_audio_cached(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y


def bass_boost_simple(y, sr, low=50, high=150, gain_db=0.0):
    if gain_db == 0.0:
        return y
    Y = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(len(y), 1 / sr)
    mask = (freqs >= low) & (freqs <= high)
    Y[mask] *= 10 ** (gain_db / 20.0)
    yb = np.fft.irfft(Y, n=len(y))
    return np.clip(yb, -1.0, 1.0)


@memory.cache
def segment_track_cached(audio_path, seg_s=15.0, sr=SR, hop_length=512, boost_db=0.0):
    y = load_audio_cached(audio_path, sr=sr)
    y = bass_boost_simple(y, sr, gain_db=boost_db)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length)
    mfcc = librosa.feature.mfcc(y=y, sr=sr)
    X = np.vstack(
        [librosa.util.normalize(chroma, axis=1), librosa.util.normalize(mfcc, axis=1)]
    )
    duration = len(y) / sr
    n_segments = max(4, int(max(1, duration / float(seg_s))))
    bound_idxs = librosa.segment.agglomerative(X, k=n_segments, axis=1)
    times = librosa.frames_to_time(bound_idxs, sr=sr, hop_length=hop_length)
    return np.unique(np.concatenate([[0.0], times, [duration]]))

In [5]:
# Features definition
def _segment_slice(y, sr, center_s, win_s, offset=0.0):
    c = center_s + offset
    start = max(0, int((c - win_s / 2) * sr))
    end = min(len(y), int((c + win_s / 2) * sr))
    seg = y[start:end]
    min_len = int(sr * 0.5)
    if len(seg) == 0:
        seg = np.zeros(min_len, dtype=np.float32)
    elif len(seg) < min_len:
        pad = min_len - len(seg)
        seg = np.pad(seg, (0, pad), mode="reflect" if len(seg) > 1 else "constant")
    return seg


def _stats_vec(x):
    if x.ndim == 1:
        return np.array(
            [np.mean(x), np.std(x), np.median(x), np.max(x), np.min(x)],
            dtype=np.float32,
        )
    m = np.mean(x, axis=1)
    s = np.std(x, axis=1)
    p = np.percentile(x, [10, 25, 50, 75, 90], axis=1)
    return np.concatenate([m, s, p.flatten()]).astype(np.float32)


@memory.cache
def features_context_cached(
    audio_path, center_s, win_s, sr=SR, hop_length=512, boost_db=0.0
):
    y = load_audio_cached(audio_path, sr=sr)
    y = bass_boost_simple(y, sr, gain_db=boost_db)
    pre = _segment_slice(y, sr, center_s, win_s, offset=-win_s / 2)
    mid = _segment_slice(y, sr, center_s, win_s, offset=0.0)
    post = _segment_slice(y, sr, center_s, win_s, offset=+win_s / 2)

    def feat(seg):
        mel = librosa.feature.melspectrogram(y=seg, sr=sr, n_mels=64, power=2.0)
        logmel = librosa.power_to_db(mel, ref=np.max)
        mfcc = librosa.feature.mfcc(S=librosa.power_to_db(mel), sr=sr, n_mfcc=13)
        onset_env = librosa.onset.onset_strength(y=seg, sr=sr, hop_length=hop_length)
        tempogram = librosa.feature.tempogram(y=seg, sr=sr, hop_length=hop_length)
        rolloff = librosa.feature.spectral_rolloff(y=seg, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=seg, sr=sr)
        zcr = librosa.feature.zero_crossing_rate(y=seg)
        rms = librosa.feature.rms(y=seg)
        S = np.abs(librosa.stft(seg, n_fft=2048, hop_length=hop_length)) ** 2
        freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
        mask = (freqs >= 50) & (freqs <= 150)
        low_pow = np.sum(S[mask], axis=0)
        return np.concatenate(
            [
                _stats_vec(logmel),
                _stats_vec(mfcc),
                _stats_vec(onset_env),
                _stats_vec(tempogram),
                _stats_vec(rolloff),
                _stats_vec(contrast),
                _stats_vec(zcr),
                _stats_vec(rms),
                _stats_vec(low_pow),
            ]
        )

    fp, fm, fpost = feat(pre), feat(mid), feat(post)
    d_pm = fpost - fp
    d_cm = fm - fp
    d_pc = fpost - fm
    return np.concatenate([fp, fm, fpost, d_pm, d_cm, d_pc]).astype(np.float32)

In [6]:
# Build dataset
def build_dataset_from_pairs(
    pairs, seg_s, win, tol_label, tol_eval, boost_db, neg_ratio=NEG_RATIO, n_jobs=-1
):
    def _build_one(audio_path, drop_s):
        boundaries = segment_track_cached(audio_path, seg_s=seg_s, boost_db=boost_db)
        d = np.abs(boundaries - drop_s)
        labels = np.zeros(len(boundaries), dtype=np.int32)
        labels[d <= tol_label] = 1
        labels[(d > tol_label) & (d <= tol_eval)] = -1
        labels[d > tol_eval] = 0

        pos_idx = np.where(labels == 1)[0]
        ign_idx = np.where(labels == -1)[0]
        neg_idx = np.where(labels == 0)[0]
        if len(pos_idx) > 0 and len(neg_idx) > len(pos_idx) * neg_ratio:
            rng = np.random.default_rng(RANDOM_SEED)
            neg_idx = np.sort(
                rng.choice(neg_idx, size=len(pos_idx) * neg_ratio, replace=False)
            )
        keep = np.sort(np.concatenate([pos_idx, neg_idx]))
        kept_boundaries = boundaries[keep]
        feats = [
            features_context_cached(audio_path, float(t), win_s=win, boost_db=boost_db)
            for t in kept_boundaries
        ]
        X = np.vstack(feats)
        y = labels[keep]
        return X, y

    results = Parallel(n_jobs=n_jobs, prefer="processes")(
        delayed(_build_one)(p, d) for p, d in pairs
    )
    X = np.vstack([r[0] for r in results])
    y = np.concatenate([r[1] for r in results])
    return X, y

In [7]:
# Scoring (here we use F1@2)
def f1_at_two(pairs, clf, seg_s, win, tol_eval, boost_db):
    f1s = []
    for path, drop_s in pairs:
        boundaries = segment_track_cached(path, seg_s=seg_s, boost_db=boost_db)
        feats = [
            features_context_cached(path, float(t), win_s=win, boost_db=boost_db)
            for t in boundaries
        ]
        if not feats:
            continue
        X = np.vstack(feats)
        scores = clf.decision_function(X)
        top2 = np.argsort(scores)[-2:]
        times = [float(boundaries[i]) for i in top2]
        ok = any(abs(t - float(drop_s)) <= tol_eval for t in times)
        tp = 1 if ok else 0
        fp = 0 if ok else 1
        fn = 0 if ok else 1
        f1s.append(2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s)) if f1s else float("nan")

In [ ]:
# Sweep through hyperparameters
SEG_S_GRID = [10.0, 12.0, 15.0, 20.0]
BOOST_GRID = [0.0, 2.0]
WIN_GRID = [4.0, 6.0, 8.0, 10.0]
TOL_LABEL_GRID = [1.0, 1.5, 2.0]
C_GRID = [0.5, 1.0, 2.0]


def sweep_leave_tracks_out(audio_files, df_sidecar, max_tracks=dev_n_tracks, n_jobs=-1):
    pairs = []
    for p in audio_files:
        if len(pairs) >= max_tracks:
            break
        comment = get_comment_for_file(p, df_sidecar)
        s = parse_mmss(comment) if comment else None
        if s is not None:
            pairs.append((str(p), float(s)))
    if len(pairs) < 6:
        raise RuntimeError(
            "Need at least 6 labeled tracks for leave-track-out splitting."
        )

    train_pairs, val_pairs = train_test_split(
        pairs, test_size=0.25, random_state=RANDOM_SEED
    )
    rows = []
    for seg_s, boost_db, win, tol_l, C in product(
        SEG_S_GRID, BOOST_GRID, WIN_GRID, TOL_LABEL_GRID, C_GRID
    ):
        X_tr, y_tr = build_dataset_from_pairs(
            train_pairs, seg_s, win, tol_l, tol_eval, boost_db, n_jobs=n_jobs
        )
        X_va, y_va = build_dataset_from_pairs(
            val_pairs, seg_s, win, tol_l, tol_eval, boost_db, n_jobs=n_jobs
        )
        clf = Pipeline(
            [
                ("scaler", StandardScaler(with_mean=True, with_std=True)),
                (
                    "svm",
                    LinearSVC(
                        C=C,
                        class_weight="balanced",
                        random_state=RANDOM_SEED,
                        max_iter=7000,
                    ),
                ),
            ]
        )
        clf.fit(X_tr, y_tr)
        f1_val = f1_at_two(val_pairs, clf, seg_s, win, tol_eval, boost_db)
        rows.append(
            {
                "seg_s": seg_s,
                "boost_db": boost_db,
                "win": win,
                "tol_label": tol_l,
                "tol_eval": tol_eval,
                "C": C,
                "F1@2_val": f1_val,
            }
        )
        print(
            f"seg={seg_s}, boost={boost_db}, win={win}, tol_label={tol_l}, C={C} | F1@2={f1_val:.3f}"
        )
    df = pd.DataFrame(rows)
    df_sorted = df.sort_values(["F1@2_val"], ascending=False)
    return df_sorted, train_pairs, val_pairs


## Tune hyperparameters

Runs the sweep on up to dev_n_tracks to pick a good config quickly.
Prints the sorted results so you can inspect winners by F1@2_val.

In [ ]:
# Uses only n dev tracks
audio_files = find_audio_files(BASE_DIR)
df_sidecar, csv_path = read_sidecar_csv(BASE_DIR)
print(f"Found {len(audio_files)} audio files. Sidecar CSV: {csv_path}")

res_sorted, train_pairs, val_pairs = sweep_leave_tracks_out(
    audio_files, df_sidecar, max_tracks=dev_n_tracks, n_jobs=-1
)
print(f"Best hyperparameters: {res_sorted.sort_values(by='F1@2_val', ascending=False)}")

Found 327 audio files. Sidecar CSV: None


/opt/miniconda3/envs/genreclf/lib/python3.11/site-packages/resampy/filters.py:32: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


KeyboardInterrupt: 

## Train model

Uses all labeled tracks

Re-scan all audio + labels (CSV or comments), report any missing labels, and build pairs_all.

- train_final_from_pairs(...): rebuilds X,y using the chosen hyper-params, trains the final LinearSVC, and saves a bundle via joblib.dump({"model": clf, "params": {...}}, "drop_detector_model.joblib").

In [ ]:
# Uses all labeled tracks
audio_files_all = find_audio_files(BASE_DIR)
df_sidecar_all, csv_path_all = read_sidecar_csv(BASE_DIR)
print(f"Found {len(audio_files_all)} audio files. Sidecar CSV: {csv_path_all}")

pairs_all = []
for p in audio_files_all:
    comment = get_comment_for_file(p, df_sidecar_all)
    s = parse_mmss(comment) if comment else None
    if s is not None:
        pairs_all.append((str(p), float(s)))
    else:
        print(f"Missing label on track:{p}")

print(f"Total labeled tracks: {len(pairs_all)}")
if len(pairs_all) == 0:
    raise RuntimeError(
        "No labeled tracks found. Ensure comments contain mm:ss or provide a CSV with a 'Comment' column."
    )

Found 327 audio files. Sidecar CSV: None
Total labeled tracks to train on: 327


In [ ]:
# Training
try:
    best = res_sorted.iloc[0].to_dict()
    print("Best (from sweep):", best)
except NameError:
    print(
        "res_sorted is undefined. Run the sweep cell first to populate the best hyperparameters."
    )


def train_final_from_pairs(
    all_pairs,
    seg_s,
    win,
    tol_label,
    tol_eval,
    boost_db,
    C,
    n_jobs=-1,
    out_path="drop_detector_model.joblib",
):
    X, y = build_dataset_from_pairs(
        all_pairs, seg_s, win, tol_label, tol_eval, boost_db, n_jobs=n_jobs
    )
    clf = Pipeline(
        [
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            (
                "svm",
                LinearSVC(
                    C=C,
                    class_weight="balanced",
                    random_state=RANDOM_SEED,
                    max_iter=7000,
                ),
            ),
        ]
    )
    clf.fit(X, y)
    joblib.dump(
        {
            "model": clf,
            "params": {
                "seg_s": seg_s,
                "win": win,
                "tol_label": tol_label,
                "tol_eval": tol_eval,
                "boost_db": boost_db,
                "C": C,
            },
        },
        out_path,
    )
    return out_path


final_model_path = train_final_from_pairs(
    all_pairs=pairs_all,
    seg_s=10,  # float(best["seg_s"]),
    win=8,  # float(best["win"]),
    tol_label=1,  # float(best["tol_label"]),
    tol_eval=3,  # float(best["tol_eval"]),
    boost_db=2,  # float(best["boost_db"]),
    C=2,  # float(best["C"]),
    n_jobs=-1,
    out_path="drop_detector_model.joblib",
)

print("Saved final model to:", final_model_path)

res_sorted is undefined. Run the sweep cell first to populate the best hyperparameters.


/opt/miniconda3/envs/genreclf/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/var/folders/f4/y03jc2rs55q9y24pqptp5dg40000gn/T/ipykernel_36466/270714120.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
/opt/miniconda3/envs/genreclf/lib/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Saved final model to: drop_detector_linear_svm_best.joblib


## Inference

- find_drop_times(audio_path, model_bundle_path="drop_detector_model.joblib")
Loads the saved bundle, recomputes segmentation + features for the track, scores all boundaries, and returns the top-2 predicted drop times with their decision scores.

In [ ]:
def find_drop_times(audio_path, model_bundle_path="drop_detector_model.joblib"):
    bundle = joblib.load(model_bundle_path)
    clf = bundle["model"]
    params = bundle["params"]
    seg_s = params["seg_s"]
    win = params["win"]
    boost_db = params["boost_db"]
    boundaries = segment_track_cached(audio_path, seg_s=seg_s, boost_db=boost_db)
    feats = [
        features_context_cached(audio_path, float(t), win_s=win, boost_db=boost_db)
        for t in boundaries
    ]
    X = np.vstack(feats)
    scores = clf.decision_function(X)
    order = np.argsort(scores)[-2:][::-1]
    return [(float(boundaries[i]), float(scores[i])) for i in order]


test_file = str(
    find_audio_files("/Users/Administrateur/Downloads/Music/Download/to dl 10 yt")[0]
)
find_drop_times(test_file)

[(107.62448979591836, 0.538387801832767),
 (40.936780045351476, 0.4235882229844672)]